In [1]:
using Pkg
Pkg.activate(".")
# using CanonicalTraits
# using Moshi
using Test

  Activating project at `~/Documents/GitHub/Julia_Tutorials/CategoryTheory`


In [21]:
using SumTypes

@sum_type Either{A, B} begin
    Left{A}(::A)
    Right{B}(::B)
end
struct A end
struct B end

function Either{A, B}(x::A) where {A, B}
    return Left(x)
end
function Either{A, B}(x::B) where {A, B}
    return Right(x)
end

function foo(x)::Either{A, B}
    if x > 2
        # return Either{A, B}(A())  # Explicitly construct Left(A())
        return Left(A())
    else
        return Right(B())
        # return Either{A, B}(B())  # Explicitly construct Right(B())
    end
end

foo(1)
@inferred foo(1)

Right(B())::Either{A, B}

In [22]:
using ExtensibleEffects
@syntax_eff begin
    a = [1, 2]
    b = ["one", "two"]
    @pure a, b
end

option_example(n) = @syntax_eff begin
     a = Option(n)
     b = @Try isodd(a) ? error("fail") : a+1
     @pure a, b
end

@inferred option_example(3)

LoadError: return type Const{Thrown{ErrorException, Base.ExceptionStack}} does not match inferred return type Any

In [ ]:
Maybe{T} = Union{T, Nothing}
Maybe(T::Type) = Union{T, Nothing}

fmap(f::Function, x::Maybe) = x === nothing ? nothing : f(x)

f(x) = x^2
@show fmap(f,nothing);
@show fmap(f,2);

In [ ]:
@inferred fmap(x -> x + 1, Some(1))
# fmap(x -> x + 1, Some(1))
# fmap(x -> x + 1, Some(1))
# @inferred fmap(x -> x + 1, Nothing())

In [ ]:
fmap(x -> x + 1, Nothing())
# Nothing::Maybe{Uninit}

In [ ]:
@code_warntype fmap(x -> x + 1, Some(3))

In [ ]:
using Moshi: Data.@data, Match.@match

@data Maybe{T} begin
    Some(T)
    Nothing
end

fmap(f, m::Maybe.Type) = @match m begin
    Maybe.Some(x) => Maybe.Some(f(x))
    Maybe.Nothing() => Maybe.Nothing{S}()
end

In [ ]:
typeof(Maybe)

In [ ]:
# @code_warntype fmap(x -> x + 1, Maybe.Some(3))
# @code_warntype map(x->x+1,[1,2])

In [ ]:
@trait Monoid{A} begin
                  mempty :: Type{A} => A
                  # a method with two arguments
                  (⊕)    :: [A, A] => A
end


@implement Monoid{Num} where Num <: Number begin
           mempty(::Type{Num}) = zero(Num)
           (a :: Num) ⊕ (b :: Num) = a + b
end